# Transcript Matching — Method & Preprocessing Evaluation
Evaluates different fuzzy-ratio combinations and preprocessing strategies (transcript cuts, accent stripping, etc.)  
across cross-validation folds. Outputs a LaTeX-ready accuracy table.

## 1. Imports & Setup

In [1]:
import os
import sys
import random
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve

module_path = os.path.abspath(os.path.join(''))
if module_path not in sys.path:
    sys.path.append(module_path)

if os.path.isdir("../data/"):
    os.chdir("../")

from src.utils import *
from src.plots_utils import *
from src.transcript_matching import *
from src.file_io import *

%load_ext autoreload
%autoreload 2

rcParams, column_width, page_width, shorts_style, RV_style = get_plt_attr()
plt.rcParams.update(rcParams)

logger = logging.getLogger(__name__)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename='logs/transcript_matching.log',
    level=logging.DEBUG,
    format='%(asctime)s:%(levelname)s:%(message)s',
    datefmt='%m-%d-%Y %I:%M:%S'
)

## 2. Load Data

In [2]:
nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')
nm_tt_channels = pd.read_json('data/tiktok/channels/news_channels.json')
pp_tt_channels = pd.read_json('data/tiktok/channels/pp_channels.json')

In [13]:
# --- transcripts ---
yt_nm_videos_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl", lines=True)
yt_nm_videos_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl", lines=True)
yt_pp_videos_2022 = pd.read_json("data/youtube/videos/pp_videos_2022.jsonl", lines=True)
yt_pp_videos_2024 = pd.read_json("data/youtube/videos/pp_videos_2024.jsonl", lines=True)

tt_nm_videos_2022 = pd.read_json("data/tiktok/videos/news_videos_2022.jsonl", lines=True)
tt_nm_videos_2024 = pd.read_json("data/tiktok/videos/news_videos_2024.jsonl", lines=True)
tt_pp_videos_2022 = pd.read_json("data/tiktok/videos/pp_videos_2022.jsonl", lines=True)
tt_pp_videos_2024 = pd.read_json("data/tiktok/videos/pp_videos_2024.jsonl", lines=True)
for tt_df in [tt_nm_videos_2022, tt_nm_videos_2024, tt_pp_videos_2022, tt_pp_videos_2024]:
    tt_df.rename(columns={'id':'videoId', 'voice_to_text':'transcript','create_time':'publishedAt'}, inplace=True)

all_transcripts = pd.concat([
    yt_nm_videos_2022[['videoId', 'transcript', 'name_standard']], yt_pp_videos_2022[['videoId', 'transcript', 'name_standard']],
    tt_nm_videos_2022[['videoId', 'transcript', 'name_standard']], tt_pp_videos_2022[['videoId', 'transcript', 'name_standard']],
    yt_nm_videos_2024[['videoId', 'transcript', 'name_standard']], yt_pp_videos_2024[['videoId', 'transcript', 'name_standard']],
    tt_nm_videos_2024[['videoId', 'transcript', 'name_standard']], tt_pp_videos_2024[['videoId', 'transcript', 'name_standard']],
]).dropna(subset='transcript')


all_transcripts['transcript_clean'] = all_transcripts['transcript'].apply(
    clean_transcripts,
    strip_accents=True, lowercase=True, remove_brackets=True, remove_openers=False
)
print(f"{len(all_transcripts):,} transcripts loaded")

91,701 transcripts loaded


In [14]:
video2channel = dict(zip(all_transcripts['videoId'], all_transcripts['name_standard']))

## 3. Prepare Labeled Pairs

In [18]:
def prepare_labeled_pairs(labeled_pairs: pd.DataFrame, transcripts: pd.DataFrame) -> pd.DataFrame:
    """Merge transcript text into a labeled-pairs dataframe and add binary label."""
    initial_length = len(labeled_pairs)
    transcripts['videoId'] = transcripts['videoId'].apply(str)
    labeled_pairs.dropna(subset="Label", inplace=True)

    valid_labels = {"No", "Yes, one whole fragment", "Yes, cut fragments", "Yes"}
    assert set(labeled_pairs.Label) - valid_labels == set(), set(labeled_pairs.Label)

    id_name = 'id' if ('id1' in labeled_pairs.columns) else 'videoId'

    def clean_id(x):
        return (x.replace('www.youtube.com/shorts/', '')
                 .replace('www.youtube.com/watch?v=', '')
                 .replace('www.tiktok.com/@user/video/', ''))

    labeled_pairs['videoId1'] = labeled_pairs[f'{id_name}1'].apply(clean_id)
    labeled_pairs['videoId2'] = labeled_pairs[f'{id_name}2'].apply(clean_id)
    labeled_pairs['id1,id2'] = labeled_pairs['videoId1'] + labeled_pairs['videoId2']

    labeled_pairs = (
        labeled_pairs
        .merge(transcripts[['videoId', 'transcript', 'transcript_clean']],
               left_on='videoId1', right_on='videoId')
        .rename(columns={'transcript_clean': 'transcript_clean1', 'transcript': 'transcript1'})
        .drop(columns=['videoId'])
    )
    labeled_pairs = (
        labeled_pairs
        .merge(transcripts[['videoId', 'transcript', 'transcript_clean']],
               left_on='videoId2', right_on='videoId')
        .rename(columns={'transcript_clean': 'transcript_clean2', 'transcript': 'transcript2'})
        .drop(columns=['videoId'])
    )

    if len(labeled_pairs) != initial_length:
        print(f"Length mismatch: {initial_length} → {len(labeled_pairs)}")

    labeled_pairs = labeled_pairs[labeled_pairs.transcript1.str.len() > 50]
    labeled_pairs = labeled_pairs[labeled_pairs.transcript2.str.len() > 50]
    labeled_pairs['label_binary'] = (labeled_pairs['Label'] != 'No').astype(int)
    return labeled_pairs

## 4. Load & Compute Ratios for Labeled Pairs

In [19]:
# YouTube-only pairs
yt_annotation_files = [
    "data/youtube/transcripts/manual_annotations/similarities_same_channel_news_2024_labeled.xlsx",
    "data/youtube/transcripts/manual_annotations/similarities_diff_channel_polit_2024_labeled.xlsx",
    "data/youtube/transcripts/manual_annotations/similarities_same_channel_polit_2024_labeled.xlsx",
]
labeled_pairs_youtube_only = pd.concat([
    prepare_labeled_pairs(pd.read_excel(f, index_col=0), all_transcripts)
    for f in yt_annotation_files
]).reset_index(drop=True)

labeled_pairs_youtube_only['transcript_cut1'] = labeled_pairs_youtube_only['transcript_clean1'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_only['transcript_cut2'] = labeled_pairs_youtube_only['transcript_clean2'].apply(
    extract_short_transcript, length_transcript=2000)

labeled_pairs_youtube_only = compute_ratios(labeled_pairs_youtube_only, '_cut_t1', 'transcript_cut1', 'transcript_clean2')
labeled_pairs_youtube_only = compute_ratios(labeled_pairs_youtube_only, '_cut_t2', 'transcript_clean1', 'transcript_cut2')

print(f"YouTube-only pairs: {len(labeled_pairs_youtube_only):,}")

Length mismatch: 65 → 64


YouTube-only pairs: 228


In [21]:
same_channels_nm = prepare_labeled_pairs(
    pd.read_excel("data/youtube/transcripts/manual_annotations/similarities_same_channel_news_2024_labeled.xlsx", index_col=0), 
    all_transcripts
)

same_channels_pp = prepare_labeled_pairs(
    pd.read_excel( "data/youtube/transcripts/manual_annotations/similarities_same_channel_polit_2024_labeled.xlsx", index_col=0), 
    all_transcripts
)

diff_channels_yt = prepare_labeled_pairs(
    pd.read_excel("data/youtube/transcripts/manual_annotations/similarities_diff_channel_polit_2024_labeled.xlsx", index_col=0), 
    all_transcripts
)

Length mismatch: 65 → 64


In [22]:

labeled_pairs_youtube_only = pd.concat([same_channels_nm, same_channels_pp, diff_channels_yt
    
]).reset_index(drop=True)

labeled_pairs_youtube_only['transcript_cut1'] = labeled_pairs_youtube_only['transcript_clean1'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_only['transcript_cut2'] = labeled_pairs_youtube_only['transcript_clean2'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_only = compute_ratios(labeled_pairs_youtube_only, '_cut_t1', 'transcript_cut1', 'transcript_clean2')
labeled_pairs_youtube_only = compute_ratios(labeled_pairs_youtube_only, '_cut_t2', 'transcript_clean1', 'transcript_cut2')



In [23]:
# YouTube × TikTok pairs
tt_annotation_files = [
    "data/pairs_of_transcripts/annotation/pairs_yt_tt_nm_2022_to_annotate.xlsx",
    "data/pairs_of_transcripts/annotation/pairs_yt_tt_pp_2024_to_annotate.xlsx",
]
labeled_pairs_youtube_tiktok = pd.concat([
    prepare_labeled_pairs(pd.read_excel(f, index_col=0), all_transcripts)
    for f in tt_annotation_files
]).reset_index(drop=True)

labeled_pairs_youtube_tiktok['transcript_cut1'] = labeled_pairs_youtube_tiktok['transcript_clean1'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_tiktok['transcript_cut2'] = labeled_pairs_youtube_tiktok['transcript_clean2'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_tiktok = compute_ratios(labeled_pairs_youtube_tiktok, '_cut_t1', 'transcript_cut1', 'transcript_clean2')
labeled_pairs_youtube_tiktok = compute_ratios(labeled_pairs_youtube_tiktok, '_cut_t2', 'transcript_clean1', 'transcript_cut2')

print(f"YouTube × TikTok pairs: {len(labeled_pairs_youtube_tiktok):,}")

YouTube × TikTok pairs: 80


## 5. Cross-Validation Evaluation

In [24]:
def split(a, n):
    """Split list `a` into `n` roughly equal parts."""
    k, m = divmod(len(a), n)
    return [a[i*k+min(i, m):(i+1)*k+min(i+1, m)] for i in range(n)]


def evaluate_preprocessing(labeled_pairs: pd.DataFrame, case2columns_dict: dict, n_folds: int = 5):
    """
    5-fold cross-validation over a set of feature column combinations.
    Returns (mean_accuracy_dict, std_accuracy_dict).
    """
    labeled_pairs = labeled_pairs[
        (labeled_pairs['transcript_clean1'].str.len() > 100) &
        (labeled_pairs['transcript_clean2'].str.len() > 100)
    ].copy()

    random.seed(0)
    list_idx = list(labeled_pairs.index)
    random.shuffle(list_idx)
    folds = split(list_idx, n_folds)

    cases_results, cases_results_stds = {}, {}

    for case, columns in case2columns_dict.items():
        print(f"  [{case}]  features: {columns}")
        Xs = {i: labeled_pairs.loc[fold, columns] for i, fold in enumerate(folds)}
        ys = {i: labeled_pairs.loc[fold, 'label_binary'] for i, fold in enumerate(folds)}

        accuracies = []
        for i in range(n_folds):
            X_train = pd.concat([Xs[j] for j in range(n_folds) if j != i])
            y_train = pd.concat([ys[j] for j in range(n_folds) if j != i])
            X_test, y_test = Xs[i], ys[i]

            model = LogisticRegression()
            model.fit(X_train, y_train)

            train_probas = model.predict_proba(X_train)[:, 1]
            prec, rec, thr = precision_recall_curve(y_train, train_probas)
            f1 = 2 * prec * rec / (prec + rec)
            best_thr = np.mean([thr[np.argmax(f1)], thr[np.argmax(f1) - 1]])

            test_probas = model.predict_proba(X_test)[:, 1]
            y_pred = (test_probas >= best_thr).astype(int)
            accuracies.append((y_pred == y_test).mean())

        cases_results[case] = np.mean(accuracies)
        cases_results_stds[case] = np.std(accuracies)

    return cases_results, cases_results_stds

In [25]:
# Feature-set definitions
case2columns_dict_both_type = {
    'partial':          ['partialratio_cut_t1', 'partialratio_cut_t2', 'diffchannel'],
    'sort':             ['sortratio_cut_t1',    'sortratio_cut_t2',    'diffchannel'],
    'set':              ['setratio_cut_t1',     'setratio_cut_t2',     'diffchannel'],
    'partial sort':     ['partialratio_cut_t1', 'partialratio_cut_t2',
                         'sortratio_cut_t1',    'sortratio_cut_t2',    'diffchannel'],
    'partial set':      ['partialratio_cut_t1', 'partialratio_cut_t2',
                         'setratio_cut_t1',     'setratio_cut_t2',     'diffchannel'],
    'sort set':         ['sortratio_cut_t1',    'sortratio_cut_t2',
                         'setratio_cut_t1',     'setratio_cut_t2',     'diffchannel'],
    'partial sort set': ['partialratio_cut_t1', 'partialratio_cut_t2',
                         'sortratio_cut_t1',    'sortratio_cut_t2',
                         'setratio_cut_t1',     'setratio_cut_t2',     'diffchannel'],
}

In [27]:
labeled_pairs

,party,videoId1,videoId2,"id1,id2",transcript1,transcript2,Label,transcript_clean1,transcript_clean2,partialratio,sortratio,setratio,channel1,channel2,transcript1_length,transcript2_length,diffchannel,label_binary
95,La France insoumise,r4KZ1a7t-Qg,dR5iZtzwgVw,www.youtube.com/shorts/r4KZ1a7t-Qgwww.youtube....,monsieur Bardella c'est le moment de faire vot...,[Musique] pour c [Applaudissements] [Musique] ...,No,monsieur Bardella c'est le moment de faire vot...,[Musique] pour c [Applaudissements] [Musique] ...,45,12,62,LFI,NaN,980,14629,True,0
154,UPR,Gq_5GZqbk7o,nkgeNVC455E,www.youtube.com/shorts/Gq_5GZqbk7owww.youtube....,bonjour à tous charlotte a cru que ça recommen...,mais Suin oui il faudrait quand même peut-être...,"Yes, one whole fragment",bonjour à tous charlotte a cru que ça recommen...,mais Suin oui il faudrait quand même peut-être...,98,9,97,Europe 1,François Asselineau,26002,1231,True,1
155,UPR,Gq_5GZqbk7o,X54sEerWnog,www.youtube.com/shorts/Gq_5GZqbk7owww.youtube....,bonjour à tous charlotte a cru que ça recommen...,mon objectif c'est d'abord et avant tout ça va...,No,bonjour à tous charlotte a cru que ça recommen...,mon objectif c'est d'abord et avant tout ça va...,48,6,79,Europe 1,François Asselineau,26002,794,True,0
156,UPR,Gq_5GZqbk7o,gu7glqmmBlk,www.youtube.com/shorts/Gq_5GZqbk7owww.youtube....,bonjour à tous charlotte a cru que ça recommen...,[Musique] et bien bonjour à toutes et à tous c...,No,bonjour à tous charlotte a cru que ça recommen...,[Musique] et bien bonjour à toutes et à tous c...,45,59,66,Europe 1,UPR,26002,50296,True,0
157,LFI,A0qaMnrgSyY,jDCXGgiDgDY,www.youtube.com/shorts/A0qaMnrgSyYwww.youtube....,il est hors de question tous les 5 ans de refa...,vous avez dit tout à l'heure à propos de ce qu...,"Yes, cut fragments",il est hors de question tous les 5 ans de refa...,vous avez dit tout à l'heure à propos de ce qu...,95,4,99,Le Média,Jean-Luc Mélenchon,53928,1031,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,Emmanuel Macron,LtE49UXYIDo,FG6DCiGrA-0,www.youtube.com/shorts/LtE49UXYIDowww.youtube....,[Musique] bonjour monsieur le Président bonjou...,[Musique] [Musique] bonjour monsieur le Présid...,Yes,[Musique] bonjour monsieur le Président bonjou...,[Musique] [Musique] bonjour monsieur le Présid...,98,98,96,Emmanuel Macron,RMC,67352,66929,True,1
233,Emmanuel Macron,fslUZZ_i4DU,22P-5Hcngiw,www.youtube.com/shorts/fslUZZ_i4DUwww.youtube....,quand on est citoyen européen je pense qu'il y...,sécurité renforcée dans le puit de d'Ô pour un...,No,quand on est citoyen européen je pense qu'il y...,sécurité renforcée dans le puit de d'Ô pour un...,50,6,87,Emmanuel Macron,Europe 1,224,7795,True,0
234,Emmanuel Macron,Xh31Xlw_CA4,x0HFA1EfAow,www.youtube.com/shorts/Xh31Xlw_CA4www.youtube....,comme vous j'ai faim et donc je pense que c'es...,française français vous avez eu ce jour à vote...,No,comme vous j'ai faim et donc je pense que c'es...,française français vous avez eu ce jour à vote...,47,21,64,Emmanuel Macron,Public Sénat,494,3894,True,0
235,Gabriel Attal,XKBlykLqZEw,MNoW6jNknT8,www.youtube.com/shorts/XKBlykLqZEwwww.youtube....,nouveau Premier ministre dans quelques jours d...,un point de détail je précise que votre place ...,Yes,nouveau Premier ministre dans quelques jours d...,un point de détail je précise que votre place ...,94,94,93,Gabriel Attal,LCI,117456,120466,True,1


In [26]:
evaluate_df_results = pd.DataFrame(
    index=['partial', 'sort', 'set', 'partial sort', 'partial set', 'sort set', 'partial sort set'])
std_df_results = evaluate_df_results.copy()

eval_dir = 'data/youtube/transcripts/evaluate/'
for filename in sorted(os.listdir(eval_dir)):
    if filename == '.DS_Store':
        continue
    print(f"\n=== {filename} ===")
    labeled_pairs = pd.read_csv(eval_dir + filename)
    labeled_pairs = process_pairs(labeled_pairs, video2channel)
    labeled_pairs = labeled_pairs[labeled_pairs.diffchannel]

    means, stds = evaluate_preprocessing(labeled_pairs, case2columns_dict_both_type)
    col = filename.replace('.csv', '')
    evaluate_df_results[col] = means
    std_df_results[col] = stds


=== labeled_pairs_all_.csv ===


  [partial]  features: ['partialratio_cut_t1', 'partialratio_cut_t2', 'diffchannel']


KeyError: "['partialratio_cut_t1', 'partialratio_cut_t2'] not in index"

## 6. Results

In [ ]:
col_order = [
    'labeled_pairs_all_', 'labeled_pairs_all_cut_2000', 'labeled_pairs_all_cut_5000',
    'labeled_pairs_all_cut_10000', 'labeled_pairs_all_cut_15000', 'labeled_pairs_all_cut_20000',
]
evaluate_df_results = evaluate_df_results[col_order]
std_df_results       = std_df_results[col_order]

evaluate_df_results.style.highlight_max(color='lightgreen', axis=0)

In [ ]:
std_df_results

### LaTeX table

In [ ]:
label_map = {
    'partial': 'pr', 'sort': 'srtr', 'set': 'str',
    'partial sort': 'pr \\& srtr', 'partial set': 'pr \\& str',
    'sort set': 'srtr \\& str', 'partial sort set': 'pr, srtr \\& str',
}
std_stars = std_df_results.map(lambda x: "$^*$" if x < 0.05 else "")
evaluate_complete = (
    evaluate_df_results.map(lambda x: f"{x:.2f}") + std_stars
).rename(index=label_map)

for i, (idx, row) in enumerate(evaluate_complete.iterrows()):
    if i in [3, 6]:
        print('\\hline')
    best = max(row.values)
    cells = [
        f"\\textbf{{{v}}}" if '0.99' in v else v
        for v in row.values
    ]
    print(idx + ' & ' + ' & '.join(cells) + '\\\\')